# SEED-VII EEG-Conformer 端到端流水线

**新增第 0 步：先把 32 个分卷在 ModelScope 上 stream-merge 成一个完整 zip 并 push 回去；之后所有逻辑都基于这个完整 zip。**

数据源三选一（按推荐顺序）：
- **C. ModelScope 单一合并 zip**（推荐）— 已合并完后用 HTTP Range 流式读取，**磁盘 0 字节**
- **B. ModelScope 多分卷** — 兼容性 fallback；不需要预合并
- **A. 本地分卷**

脚本：
0\. `scripts/merge_and_upload.py` — **在 ModelScope 上 stream-merge** 32 分卷为 1 个 zip 并 push 回（任意时刻 ≤ 1 卷 + 16 MB 内存）
1\. `scripts/preprocess_to_npz.py` — 流式预处理 → npz（含 trial-level 划分）
2\. `scripts/train.py` — 训练 / 续训（断点续训 + 10h 软超时）
3\. `scripts/encode.py` — 编码导出 embedding
4\. `scripts/upload_to_modelscope.py` — 上传其它产物

辅助：`scripts/ms_fetch.py` — `list / fetch-info / fetch-one / fetch-volumes`。

> 严格遵循 `Design.md` 的所有原则。

## 0. 环境与路径

In [ ]:
import os, sys, subprocess, json, pathlib

REPO = pathlib.Path('/mnt/workspace/EEG_encoder_version2').resolve()
assert (REPO / 'scripts' / 'preprocess_to_npz.py').is_file(), \
 f'Repo not found at {REPO}'
print('REPO =', REPO)
sys.path.insert(0, str(REPO))

# ====== 数据源选择 (ModelScope预处理MAT方案) ======
USE_MERGED_ZIP = False  # 不再合并
USE_MULTI_VOLUMES = False
USE_LOCAL_VOLUMES = False
USE_MAT_FROM_MS = True

# ====== ModelScope配置 ======
MS_DATASET = 'DEREKVERSE/SEED-VII'
MS_REVISION = 'master'
MS_TOKEN = os.environ.get('MODELSCOPE_API_TOKEN', 'ms-2460c377-dcfc-4cb5-86d5-635cfa6ea9e2')

# 远程路径
MAT_REMOTE_GLOB = 'EEG_preprocessed/*.mat'
SAVE_INFO_GLOB = 'save_info/*_save_info.csv'

# 本地工作区
WORK = pathlib.Path('/mnt/workspace/seed_vii_work')
WORK.mkdir(parents=True, exist_ok=True)
MAT_LOCAL_DIR = str(WORK / 'mat_files')
SAVE_INFO_LOCAL = str(WORK / 'save_info')
MS_SCRATCH = str(WORK / '_ms_cache')

NPZ_OUT = str(WORK / 'preprocessed' / 'seed_vii.npz')
RUN_DIR = str(WORK / 'runs_seed_vii')
ENC_OUT = str(WORK / 'encoded' / 'embeddings.npz')

# 数据集内托管路径
NPZ_REPO_PATH = 'artifacts/preprocessed/seed_vii.npz'

def run(cmd, env=None):
    print('$', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=str(REPO), env={**os.environ, **(env or {})})
    if res.returncode != 0:
        raise SystemExit(f'cmd failed with code {res.returncode}')

PY = sys.executable
print('python =', PY)


必须登陆，否则鉴权不通过

In [ ]:
import os
# 把这个 token 换成你自己的 SDK access token
os.environ['MODELSCOPE_API_TOKEN'] = 'ms-2460c377-dcfc-4cb5-86d5-635cfa6ea9e2'

# 立即 login，把 cookie 写到本地缓存（subprocess 才能读到）
from modelscope.hub.api import HubApi
HubApi().login(os.environ['MODELSCOPE_API_TOKEN'])
print('login done')


## 0. 全面并发 Stream-merge 32 分卷 → 1 个 zip 并 push 回 ModelScope（**全面并发版**）

相比旧版“下载 1 个 → 上传 1 个 → 删除 1 个”的完全串行流程，这里使用 **多线程池** 让多个分卷同时流转：

- 每个线程独立：下载分卷 → 作为对应 OSS Part 上传 → 立即删除本地缓存。
- 多个下载、多个上传 **同时进行**，网络带宽上下行同时打满。
- 任意时刻本地最多保留 `MERGE_UPLOAD_WORKERS` 个分卷（默认 4 ≈ 22 GB），100 GB 持久盘安全。
- 仍支持 **断点续传**：已成功的 Part 会写入 state JSON，重跑自动跳过。

> 底层使用 `src/oss_merge_concurrent.py`，基于阿里云 OSS Multipart Upload 协议，天然支持乱序 Part 上传。

In [ ]:
# Stage 0 已跳过：使用预处理MAT，无需合并32分卷
print('✓ 跳过zip合并，直接使用ModelScope上的MAT文件')


## 1. 下载预处理MAT + 流式预处理 → npz

**新方案**：不再合并32分卷，直接使用ModelScope上预处理的20个MAT文件。
标签文件名如 `1_20221005_2_save_info.csv` 已在 `src/dataset.py` 中增强解析（subject=1, session=2）。


In [ ]:
# ====== 1. 从ModelScope下载MAT和save_info ======
from pathlib import Path
from src.ms_download import download_files, list_dataset_files, filter_files, login_if_token

login_if_token(MS_TOKEN)
Path(MAT_LOCAL_DIR).mkdir(parents=True, exist_ok=True)
Path(SAVE_INFO_LOCAL).mkdir(parents=True, exist_ok=True)

print('[MS] 列出远程文件...')
all_files = list_dataset_files(MS_DATASET, revision=MS_REVISION, token=MS_TOKEN)
mat_files = [f for f in all_files if f['Path'].lower().endswith('.mat') and 'eeg_preprocessed' in f['Path'].lower()]
save_files = [f for f in all_files if f['Path'].lower().endswith('_save_info.csv')]

# 如果上面过滤为空，尝试通配
if not mat_files:
    mat_files = filter_files(all_files, include=[MAT_REMOTE_GLOB, '*.mat'])
if not save_files:
    save_files = filter_files(all_files, include=[SAVE_INFO_GLOB])

mat_paths = [f['Path'] for f in mat_files]
save_paths = [f['Path'] for f in save_files]

print(f'找到 {len(mat_paths)} 个 .mat, {len(save_paths)} 个 save_info')

# 下载
download_files(MS_DATASET, mat_paths, MAT_LOCAL_DIR, revision=MS_REVISION, token=MS_TOKEN, verbose=True)
download_files(MS_DATASET, save_paths, SAVE_INFO_LOCAL, revision=MS_REVISION, token=MS_TOKEN, verbose=True)

print(f'✓ MAT -> {MAT_LOCAL_DIR}')
print(f'✓ save_info -> {SAVE_INFO_LOCAL}')

# 验证标签解析
from src.dataset import load_save_info_intensity
intensities = load_save_info_intensity(SAVE_INFO_LOCAL)
print(f'[验证] 连续标签数量: {len(intensities)} (期望1600)')

# ====== 2. 预处理 → npz ======
common = ['--output', NPZ_OUT,
          '--val-ratio', '0.1', '--test-ratio', '0.1',
          '--split-unit', 'trial',
          '--max-windows-per-trial', '40',     # 60 → 40，磁盘上限 ~12 GB；模型容量足够
          '--tmp-dir', str(WORK / 'preprocessed'),  # 放到工作盘
          # 不加 --compress，节省 RAM
          ]

run([PY, 'scripts/preprocess_to_npz.py',
     '--mat-dir', MAT_LOCAL_DIR,
     '--mat-pattern', '*.mat',
     '--save-info-dir', SAVE_INFO_LOCAL,
     *common])


## 1.5 上传预处理结果 npz 到 ModelScope（供后续实例 / 续训直接拉取）

把耗时最长的预处理产物（`seed_vii.npz`）传回数据集仓库，这样：
- 新启动的实例不需要重新执行预处理。
- 后续训练阶段可以从数据集直接下载这个 npz 开始训练。

In [ ]:
from pathlib import Path
from src.ms_upload import upload_file_to_dataset

npz_local = Path(NPZ_OUT)
if npz_local.exists():
    upload_file_to_dataset(
        local_file=NPZ_OUT,
        dataset_id=MS_DATASET,
        path_in_repo=NPZ_REPO_PATH,
        token=MS_TOKEN,
        commit_message='upload preprocessed seed_vii.npz',
    )
    print(f'[OK] Uploaded {NPZ_OUT} -> {MS_DATASET}:{NPZ_REPO_PATH}')
else:
    print(f'[WARN] {NPZ_OUT} not found; skipping upload.')


## 1.8 确保训练数据存在（本地优先，缺失时自动从 ModelScope 拉取）

在真正开始训练前，检查本地 `seed_vii.npz` 是否存在。如果不存在（例如新实例、磁盘被清理），
自动从 ModelScope 数据集下载。这让 Notebook / 命令行都能**无感恢复训练环境**。

In [ ]:
from pathlib import Path
from src.ms_download import download_one_file, login_if_token

npz_local = Path(NPZ_OUT)
if not npz_local.exists():
    print(f'Local npz not found. Downloading from {MS_DATASET}:{NPZ_REPO_PATH} ...')
    npz_local.parent.mkdir(parents=True, exist_ok=True)
    login_if_token(MS_TOKEN)
    downloaded = download_one_file(
        dataset_id=MS_DATASET,
        file_path=NPZ_REPO_PATH,
        local_dir=str(npz_local.parent),
        revision=MS_REVISION,
        token=MS_TOKEN,
    )
    # download_one_file saves as local_dir / basename(file_path)
    downloaded_path = Path(downloaded)
    if downloaded_path.resolve() != npz_local.resolve() and downloaded_path.exists():
        downloaded_path.rename(npz_local)
    if npz_local.exists():
        print(f'[OK] Downloaded to {NPZ_OUT} ({npz_local.stat().st_size / 1024**3:.2f} GB)')
    else:
        raise FileNotFoundError(f'Download failed: expected {NPZ_OUT}')
else:
    print(f'[OK] Using local npz: {NPZ_OUT} ({npz_local.stat().st_size / 1024**3:.2f} GB)')


## 2. 训练（先分类预训练，再联合训练；10h 自动安全保存）

In [ ]:
# 训练脚本已内置 --ms-data / --ms-data-path 参数作为 fallback；
# Notebook 里我们在上一 cell 已经确保本地文件存在，所以直接走本地路径即可。
run([PY, 'scripts/train.py',
     '--data', NPZ_OUT,
     '--output-dir', RUN_DIR,
     '--device', 'auto', '--amp',
     '--pretrain-epochs', '10',
     '--max-epochs', '200',
     '--batch-size', '64',
     '--lr', '2e-4', '--min-lr', '1e-5',
     '--sample-weight-mode', 'continuous',
     '--max-runtime-hours', '10'])


In [ ]:
# 续训（即使本地 npz 被清理，也会通过 --ms-data 自动拉取）
# run([PY, 'scripts/train.py',
#      '--data', NPZ_OUT,
#      '--output-dir', RUN_DIR,
#      '--device', 'auto', '--amp', '--resume',
#      '--max-runtime-hours', '10',
#      '--ms-data', MS_DATASET,
#      '--ms-data-path', NPZ_REPO_PATH,
#      '--ms-revision', MS_REVISION])


## 3. 编码导出

In [ ]:
run([PY, 'scripts/encode.py',
     '--data', NPZ_OUT,
     '--checkpoint', str(pathlib.Path(RUN_DIR) / 'best_encoder.pt'),
     '--output', ENC_OUT,
     '--feature-type', 'projected',
     '--device', 'auto', '--amp'])
import numpy as np; z = np.load(ENC_OUT, allow_pickle=True)
print({k: getattr(z[k], 'shape', z[k]) for k in z.files})


## 4. 并发上传其它产物到 ModelScope

如果需要把模型权重、编码特征等一并传回数据集，可用下面的多线程并发上传器。  
（每个文件仍走独立 commit；若文件数很多建议打包后再传。）

In [ ]:
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# 确保 src 可导入
sys.path.insert(0, str(REPO))
from src.ms_upload import upload_file_to_dataset, upload_folder_to_dataset

artifacts = [
    # (local_path, repo_path, is_folder)
    (str(Path(RUN_DIR) / 'best_encoder.pt'), 'artifacts/models/best_encoder.pt', False),
    (ENC_OUT,             'artifacts/encoded/embeddings.npz',   False),
]

def _upload_one(local_path, repo_path, is_folder):
    try:
        if is_folder:
            upload_folder_to_dataset(
                local_dir=local_path,
                dataset_id=MS_DATASET,
                path_in_repo=repo_path,
                token=MS_TOKEN,
                commit_message=f'upload {repo_path}',
            )
        else:
            upload_file_to_dataset(
                local_file=local_path,
                dataset_id=MS_DATASET,
                path_in_repo=repo_path,
                token=MS_TOKEN,
                commit_message=f'upload {repo_path}',
            )
        return (repo_path, 'ok')
    except Exception as e:
        return (repo_path, 'error', e)

print(f'Uploading {len(artifacts)} artifacts concurrently ...')
with ThreadPoolExecutor(max_workers=4) as exe:
    futures = {exe.submit(_upload_one, lp, rp, isf): rp for lp, rp, isf in artifacts}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Upload artifacts'):
        rp = futures[fut]
        try:
            res = fut.result()
            if res[1] == 'ok':
                print(f' [OK]  {rp}')
            else:
                print(f' [FAIL] {rp}: {res[2]}')
        except Exception as e:
            print(f' [FAIL] {rp}: {e}')
